# PRISM Monthly Climate Cleaning

Clean monthly PRISM precipitation and mean temperature data for the Salinas Valley / Monterey Bay HUC8 project.

This notebook:
1. Loads the PRISM export
2. Removes the extra watershed centroid point (`P001`)
3. Standardizes column names and dates
4. Checks missing values and duplicates
5. Creates one spatial layer of unique PRISM grid points
6. Saves clean outputs


## 1. Imports


In [1]:
import pandas as pd
import geopandas as gpd


## 2. Load PRISM data


In [2]:
# PRISM exports include 10 metadata rows before the table
prism = pd.read_csv(
    "PRISM_ppt_tmean_stable_4km_201301_202603.csv",
    skiprows=10
)

prism.head()


,Name,Longitude,Latitude,Elevation (ft),Date,ppt (inches),tmean (degrees F)
0,P001,-121.0237,36.0315,1063,2013-01,0.71,45.9
1,P001,-121.0237,36.0315,1063,2013-02,0.26,47.8
2,P001,-121.0237,36.0315,1063,2013-03,0.53,55.5
3,P001,-121.0237,36.0315,1063,2013-04,0.14,59.7
4,P001,-121.0237,36.0315,1063,2013-05,0.04,63.3


## 3. Keep only the regular watershed grid


In [3]:
# Remove the extra watershed centroid point
prism = prism[
    prism["Name"] != "P001"
].copy()

prism.shape


(55014, 7)

## 4. Standardize column names


In [4]:
# Rename columns for easier analysis
prism = prism.rename(columns={
    "Name": "point_id",
    "Longitude": "longitude",
    "Latitude": "latitude",
    "Elevation (ft)": "elevation_ft",
    "Date": "date",
    "ppt (inches)": "precip_in",
    "tmean (degrees F)": "tmean_f"
})

prism.head()


,point_id,longitude,latitude,elevation_ft,date,precip_in,tmean_f
159,P002,-121.7458,36.5448,919,2013-01,1.38,48.8
160,P002,-121.7458,36.5448,919,2013-02,0.84,47.9
161,P002,-121.7458,36.5448,919,2013-03,0.76,53.6
162,P002,-121.7458,36.5448,919,2013-04,0.35,56.3
163,P002,-121.7458,36.5448,919,2013-05,0.02,58.7


## 5. Convert date


In [5]:
# Convert monthly date to datetime
prism["date"] = pd.to_datetime(
    prism["date"],
    format="%Y-%m"
)

prism["date"].min(), prism["date"].max()


(Timestamp('2013-01-01 00:00:00'), Timestamp('2026-03-01 00:00:00'))

## 6. Add year and month


In [6]:
# Add simple time fields for later summaries and joins
prism["year"] = prism["date"].dt.year
prism["month"] = prism["date"].dt.month

prism.head()


,point_id,longitude,latitude,elevation_ft,date,precip_in,tmean_f,year,month
159,P002,-121.7458,36.5448,919,2013-01-01,1.38,48.8,2013,1
160,P002,-121.7458,36.5448,919,2013-02-01,0.84,47.9,2013,2
161,P002,-121.7458,36.5448,919,2013-03-01,0.76,53.6,2013,3
162,P002,-121.7458,36.5448,919,2013-04-01,0.35,56.3,2013,4
163,P002,-121.7458,36.5448,919,2013-05-01,0.02,58.7,2013,5


## 7. Quality checks


In [7]:
# Number of unique PRISM grid points
prism["point_id"].nunique()


346

In [8]:
# Check temporal coverage by point
prism.groupby("point_id")["date"].agg(
    ["min", "max", "count"]
).head()


,min,max,count
point_id,,,
P002,2013-01-01,2026-03-01,159
P003,2013-01-01,2026-03-01,159
P004,2013-01-01,2026-03-01,159
P005,2013-01-01,2026-03-01,159
P006,2013-01-01,2026-03-01,159


In [9]:
# Check missing values
prism.isna().sum()


point_id        0
longitude       0
latitude        0
elevation_ft    0
date            0
precip_in       0
tmean_f         0
year            0
month           0
dtype: int64

In [10]:
# Check duplicate point-month records
prism.duplicated(
    subset=["point_id", "date"]
).sum()


np.int64(0)

In [11]:
# Check basic value ranges
prism[
    ["precip_in", "tmean_f", "elevation_ft"]
].describe()


,precip_in,tmean_f,elevation_ft
count,55014.000000,55014.000000,55014.000000
mean,1.441166,60.100005,1378.771676
std,2.660624,9.244528,734.418049
min,0.000000,36.000000,39.000000
25%,0.000000,52.100000,938.000000
50%,0.200000,58.600000,1316.000000
75%,1.740000,68.700000,1729.000000
max,42.570000,84.500000,3934.000000


## 8. Create unique spatial grid points


In [12]:
# Create one spatial point per PRISM grid cell
prism_points = (
    prism
    .drop_duplicates(subset="point_id")
    [
        [
            "point_id",
            "longitude",
            "latitude",
            "elevation_ft"
        ]
    ]
    .copy()
)

prism_points_gdf = gpd.GeoDataFrame(
    prism_points,
    geometry=gpd.points_from_xy(
        prism_points["longitude"],
        prism_points["latitude"]
    ),
    crs="EPSG:4326"
)

prism_points_gdf.shape


(346, 5)

## 9. Save clean outputs


In [13]:
# Save cleaned monthly climate table
prism.to_csv(
    "prism_climate_monthly_clean.csv",
    index=False
)


In [14]:
# Save unique PRISM grid points for GIS
prism_points_gdf.to_file(
    "prism_points.gpkg",
    layer="prism_points",
    driver="GPKG"
)
